In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

def find_workspace_root(start=None):
    """从当前目录向上寻找项目根目录。"""
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "output" / "day04_project" / "ecommerce_customer_cleaned.csv").exists():
            return candidate
    raise FileNotFoundError("未找到项目根目录，请确认第4天清洗结果已生成。")

ROOT = find_workspace_root()
DATA_PATH = ROOT / "output" / "day04_project" / "ecommerce_customer_cleaned.csv"
OUTPUT_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("项目根目录：", ROOT)
print("输入数据：", DATA_PATH)
print("输出目录：", OUTPUT_DIR)

项目根目录： C:\Users\21975\PyCharmMiscProject
输入数据： C:\Users\21975\PyCharmMiscProject\output\day04_project\ecommerce_customer_cleaned.csv
输出目录： C:\Users\21975\PyCharmMiscProject\output\day05_analysis


In [3]:
# 读取数据（使用相对路径，通过 ROOT 定位）
DATA_PATH = ROOT / "output" / "day04_project" / "ecommerce_customer_cleaned.csv"
df = pd.read_csv(DATA_PATH)

core_cols = [
    "CustomerID", "Churn", "Tenure", "TenureGroup", "OrderCount",
    "CouponUsed", "CashbackAmount", "DaySinceLastOrder", "Complain",
    "PreferedOrderCat", "PreferredPaymentMode"
]

validation = pd.Series({
    "行数": len(df),
    "列数": df.shape[1],
    "CustomerID重复数": int(df["CustomerID"].duplicated().sum()),
    "核心字段缺失数": int(df[core_cols].isna().sum().sum()),
    "Churn取值": sorted(df["Churn"].unique().tolist()),
}, name="验证结果")

display(validation.to_frame())
display(df.head())

assert df.shape == (5630, 22), f"数据形状与第4天交付物不一致，实际为 {df.shape}"
assert df["CustomerID"].is_unique, "CustomerID存在重复"
assert df[core_cols].notna().all().all(), "核心字段仍有缺失"
assert set(df["Churn"].unique()) == {0, 1}, "Churn取值只包含0和1"
print("✅ 数据验证通过，一行代表一名用户。")


,验证结果
行数,5630
列数,22
CustomerID重复数,0
核心字段缺失数,0
Churn取值,"[0, 1]"


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,TenureGroup,IsMobileLogin
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,0-6个月,1
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,7-12个月,1
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,7-12个月,1
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,新用户,1
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,新用户,1


✅ 数据验证通过，一行代表一名用户。


In [82]:
metric_dictionary = pd.DataFrame([
    ["用户数", "CustomerID", "nunique", "独立用户数量"],
    ["流失人数", "Churn", "sum", "Churn=1的用户数量"],
    ["流失率", "Churn", "mean", "当前分组中流失用户占比"],
    ["平均订单数", "OrderCount", "mean", "用户级平均订单次数"],
    ["平均优惠券数", "CouponUsed", "mean", "用户级平均优惠券使用次数"],
    ["平均返现", "CashbackAmount", "mean", "返现金额，不等于消费金额"],
    ["平均距上次下单天数", "DaySinceLastOrder", "mean", "越大通常表示近期活跃度越低"],
], columns=["指标名称", "字段", "聚合方式", "解释边界"])
display(metric_dictionary)


,指标名称,字段,聚合方式,解释边界
0,用户数,CustomerID,nunique,独立用户数量
1,流失人数,Churn,sum,Churn=1的用户数量
2,流失率,Churn,mean,当前分组中流失用户占比
3,平均订单数,OrderCount,mean,用户级平均订单次数
4,平均优惠券数,CouponUsed,mean,用户级平均优惠券使用次数
5,平均返现,CashbackAmount,mean,返现金额，不等于消费金额
6,平均距上次下单天数,DaySinceLastOrder,mean,越大通常表示近期活跃度越低


In [83]:
overall_metrics = pd.DataFrame({
    "指标": ["用户数", "流失人数", "流失率", "平均订单数", "订单数中位数", "平均优惠券数", "平均返现", "平均App时长", "平均满意度", "平均距上次下单天数"],
    "数值": [
        df["CustomerID"].nunique(),
        df["Churn"].sum(),
        df["Churn"].mean(),
        df["OrderCount"].mean(),
        df["OrderCount"].median(),
        df["CouponUsed"].mean(),
        df["CashbackAmount"].mean(),
        df["HourSpendOnApp"].mean(),
        df["SatisfactionScore"].mean(),
        df["DaySinceLastOrder"].mean(),
    ]
})
display(overall_metrics)
print(f"总体流失率：{df['Churn'].mean():.2%}")

,指标,数值
0,用户数,"5,630.00"
1,流失人数,948.00
2,流失率,0.17
3,平均订单数,2.96
4,订单数中位数,2.00
5,平均优惠券数,1.72
6,平均返现,177.22
7,平均App时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.46


总体流失率：16.84%


In [84]:
profile_fields = ["TenureGroup", "PreferedOrderCat", "PreferredPaymentMode", "PreferredLoginDevice", "CityTier"]
for field in profile_fields:
    table = df[field].value_counts(dropna=False).rename("用户数").to_frame()
    table["用户占比"] = table["用户数"] / len(df)
    print(f"\n--- {field} ---")
    display(table)


--- TenureGroup ---


,用户数,用户占比
TenureGroup,,
0-6个月,1642,0.29
7-12个月,1584,0.28
13-24个月,1467,0.26
新用户,508,0.09
24个月以上,429,0.08



--- PreferedOrderCat ---


,用户数,用户占比
PreferedOrderCat,,
Mobile Phone,2080,0.37
Laptop & Accessory,2050,0.36
Fashion,826,0.15
Grocery,410,0.07
Others,264,0.05



--- PreferredPaymentMode ---


,用户数,用户占比
PreferredPaymentMode,,
Debit Card,2314,0.41
Credit Card,1774,0.32
E wallet,614,0.11
Cash on Delivery,514,0.09
UPI,414,0.07



--- PreferredLoginDevice ---


,用户数,用户占比
PreferredLoginDevice,,
Mobile Phone,3996,0.71
Computer,1634,0.29



--- CityTier ---


,用户数,用户占比
CityTier,,
1,3666,0.65
3,1722,0.31
2,242,0.04


In [85]:
tenure_analysis = (
    df.groupby("TenureGroup", observed=True)
      .agg(
          用户数=("CustomerID", "nunique"),
          流失人数=("Churn", "sum"),
          流失率=("Churn", "mean"),
          平均订单数=("OrderCount", "mean"),
          平均返现=("CashbackAmount", "mean"),
          平均距上次下单天数=("DaySinceLastOrder", "mean"),
      )
      .reset_index()
)
display(tenure_analysis)

,TenureGroup,用户数,流失人数,流失率,平均订单数,平均返现,平均距上次下单天数
0,0-6个月,1642,425,0.26,2.68,164.87,4.05
1,13-24个月,1467,95,0.06,3.70,204.92,5.32
2,24个月以上,429,0,0.00,3.55,222.34,5.26
3,7-12个月,1584,156,0.10,2.75,163.31,4.38
4,新用户,508,272,0.54,1.89,142.44,2.88


In [86]:
complain_analysis = (
    df.groupby("Complain")
      .agg(
          用户数=("CustomerID", "nunique"),
          流失人数=("Churn", "sum"),
          流失率=("Churn", "mean"),
          平均满意度=("SatisfactionScore", "mean"),
          平均订单数=("OrderCount", "mean"),
      )
      .reset_index()
)
complain_analysis["投诉状态"] = complain_analysis["Complain"].map({0: "无投诉", 1: "有投诉"})
display(complain_analysis[["投诉状态", "用户数", "流失人数", "流失率", "平均满意度", "平均订单数"]])

,投诉状态,用户数,流失人数,流失率,平均满意度,平均订单数
0,无投诉,4026,440,0.11,3.09,3.00
1,有投诉,1604,508,0.32,3.00,2.86


In [87]:
category_analysis = (
    df.groupby("PreferedOrderCat")
      .agg(
          用户数=("CustomerID", "nunique"),
          流失率=("Churn", "mean"),
          平均订单数=("OrderCount", "mean"),
          平均优惠券数=("CouponUsed", "mean"),
          平均返现=("CashbackAmount", "mean"),
      )
      .reset_index()
      .sort_values(["流失率", "用户数"], ascending=[False, False])
)
category_analysis["用户占比"] = category_analysis["用户数"] / len(df)
display(category_analysis)

,PreferedOrderCat,用户数,流失率,平均订单数,平均优惠券数,平均返现,用户占比
3,Mobile Phone,2080,0.27,2.18,1.37,140.20,0.37
0,Fashion,826,0.15,3.87,2.32,210.41,0.15
2,Laptop & Accessory,2050,0.10,2.77,1.65,167.22,0.36
4,Others,264,0.08,5.25,2.33,304.56,0.05
1,Grocery,410,0.05,4.60,2.19,266.23,0.07


In [88]:
payment_analysis = (
    df.groupby("PreferredPaymentMode")
      .agg(
          用户数=("CustomerID", "nunique"),
          流失率=("Churn", "mean"),
          平均订单数=("OrderCount", "mean"),
          平均优惠券数=("CouponUsed", "mean"),
          平均返现=("CashbackAmount", "mean"),
      )
      .reset_index()
      .sort_values("用户数", ascending=False)
)
display(payment_analysis)

,PreferredPaymentMode,用户数,流失率,平均订单数,平均优惠券数,平均返现
2,Debit Card,2314,0.15,2.94,1.72,177.06
1,Credit Card,1774,0.14,3.05,1.68,177.25
3,E wallet,614,0.23,3.01,1.76,185.83
0,Cash on Delivery,514,0.25,3.01,1.82,169.87
4,UPI,414,0.17,2.57,1.70,174.41


In [89]:
churn_behavior = (
    df.groupby("Churn")
      .agg(
          用户数=("CustomerID", "nunique"),
          平均订单数=("OrderCount", "mean"),
          平均优惠券数=("CouponUsed", "mean"),
          平均返现=("CashbackAmount", "mean"),
          平均App时长=("HourSpendOnApp", "mean"),
          平均满意度=("SatisfactionScore", "mean"),
          平均距上次下单天数=("DaySinceLastOrder", "mean"),
      )
      .reset_index()
)
churn_behavior["用户状态"] = churn_behavior["Churn"].map({0: "未流失", 1: "已流失"})
display(churn_behavior.drop(columns="Churn"))

,用户数,平均订单数,平均优惠券数,平均返现,平均App时长,平均满意度,平均距上次下单天数,用户状态
0,4682,2.99,1.72,180.64,2.93,3.00,4.71,未流失
1,948,2.81,1.71,160.37,2.96,3.39,3.22,已流失


In [90]:
tenure_complain_analysis = (
    df.groupby(["TenureGroup", "Complain"], observed=True)
      .agg(
          用户数=("CustomerID", "nunique"),
          流失人数=("Churn", "sum"),
          流失率=("Churn", "mean"),
          平均订单数=("OrderCount", "mean"),
      )
      .reset_index()
)
tenure_complain_analysis["投诉状态"] = tenure_complain_analysis["Complain"].map({0: "无投诉", 1: "有投诉"})
tenure_complain_analysis["样本提示"] = np.where(tenure_complain_analysis["用户数"] < 30, "小样本", "可观察")
display(tenure_complain_analysis)

,TenureGroup,Complain,用户数,流失人数,流失率,平均订单数,投诉状态,样本提示
0,0-6个月,0,1177,189,0.16,2.61,无投诉,可观察
1,0-6个月,1,465,236,0.51,2.87,有投诉,可观察
2,13-24个月,0,1053,43,0.04,3.85,无投诉,可观察
3,13-24个月,1,414,52,0.13,3.35,有投诉,可观察
4,24个月以上,0,304,0,0.00,3.75,无投诉,可观察
5,24个月以上,1,125,0,0.00,3.06,有投诉,可观察
6,7-12个月,0,1178,75,0.06,2.78,无投诉,可观察
7,7-12个月,1,406,81,0.20,2.67,有投诉,可观察
8,新用户,0,314,133,0.42,1.75,无投诉,可观察
9,新用户,1,194,139,0.72,2.12,有投诉,可观察


In [91]:
count_pivot = pd.pivot_table(
    df, index="TenureGroup", columns="Complain", values="CustomerID",
    aggfunc="nunique", fill_value=0, observed=True
).rename(columns={0: "无投诉用户数", 1: "有投诉用户数"})

churn_pivot = pd.pivot_table(
    df, index="TenureGroup", columns="Complain", values="Churn",
    aggfunc="mean", observed=True
).rename(columns={0: "无投诉流失率", 1: "有投诉流失率"})

cross_pivot = count_pivot.join(churn_pivot).reset_index()
display(cross_pivot)

Complain,TenureGroup,无投诉用户数,有投诉用户数,无投诉流失率,有投诉流失率
0,0-6个月,1177,465,0.16,0.51
1,13-24个月,1053,414,0.04,0.13
2,24个月以上,304,125,0.00,0.00
3,7-12个月,1178,406,0.06,0.20
4,新用户,314,194,0.42,0.72


In [92]:
outputs = {
    "overall_metrics.csv": overall_metrics,
    "tenure_analysis.csv": tenure_analysis,
    "complain_analysis.csv": complain_analysis,
    "category_analysis.csv": category_analysis,
    "payment_analysis.csv": payment_analysis,
    "tenure_complain_analysis.csv": tenure_complain_analysis,
    "tenure_complain_pivot.csv": cross_pivot,
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    check = pd.read_csv(path)
    print(f"已输出 {filename}: {check.shape}")

已输出 overall_metrics.csv: (10, 2)
已输出 tenure_analysis.csv: (5, 7)
已输出 complain_analysis.csv: (2, 7)
已输出 category_analysis.csv: (5, 7)
已输出 payment_analysis.csv: (5, 6)
已输出 tenure_complain_analysis.csv: (10, 8)
已输出 tenure_complain_pivot.csv: (5, 5)


In [93]:
# 任务0：项目配置（指定路径版本）
from pathlib import Path
import pandas as pd
import numpy as n
from pathlib import Path
import pandas as pd

# 直接指定完整路径
DATA_PATH = Path(r"/day04_project/day04_project\ecommerce_customer_cleaned.csv")
OUTPUT_DIR = Path(r"C:\Users\21975\PyCharmMiscProject\output\day05_analysis")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("输入:", DATA_PATH)
print("输出:", OUTPUT_DIR)

# 验证文件存在
if not DATA_PATH.exists():
    # 尝试在项目目录下搜索
    project_root = Path(r"C:\Users\21975\PyCharmMiscProject")
    found_files = list(project_root.glob("**/ecommerce_customer_cleaned.csv"))
    if found_files:
        DATA_PATH = found_files[0]
        print(f"📂 通过搜索找到文件: {DATA_PATH}")
    else:
        raise FileNotFoundError("未找到 ecommerce_customer_cleaned.csv")

print(f"📂 数据文件加载成功！")
df = pd.read_csv(DATA_PATH)
print(f"数据形状: {df.shape}")

输入: \day04_project\day04_project\ecommerce_customer_cleaned.csv
输出: C:\Users\21975\PyCharmMiscProject\output\day05_analysis
📂 通过搜索找到文件: C:\Users\21975\PyCharmMiscProject\output\day04_project\ecommerce_customer_cleaned.csv
📂 数据文件加载成功！
数据形状: (5630, 22)


In [94]:
# 任务1：加载并验收数据（必做）

# TODO 1：读取清洗后的CSV
df = pd.read_csv(DATA_PATH)

# TODO 2：输出shape、前5行和字段类型
print("数据形状：", df.shape)
print("\n前5行：")
display(df.head())
print("\n字段类型：")
display(df.dtypes)

# TODO 3：计算验收结果
core_cols = [
    "CustomerID", "Churn", "Tenure", "TenureGroup", "OrderCount",
    "CouponUsed", "CashbackAmount", "DaySinceLastOrder", "Complain",
    "PreferedOrderCat", "PreferredPaymentMode"
]

# 检查字段是否存在
existing_cols = [col for col in core_cols if col in df.columns]
missing_cols = [col for col in core_cols if col not in df.columns]

if missing_cols:
    print(f"\n⚠️ 注意：以下核心字段不存在: {missing_cols}")

validation = {
    "行数": len(df),
    "列数": df.shape[1],
    "CustomerID重复数": int(df["CustomerID"].duplicated().sum()),
    "核心字段缺失数": int(df[existing_cols].isna().sum().sum()),
    "Churn取值": sorted(df["Churn"].unique().tolist()),
}

print("\n验收结果：")
display(pd.DataFrame([validation]).T)

# 数据粒度说明
print("\n数据粒度：一行代表一名用户，每个CustomerID唯一对应一位用户。")

数据形状： (5630, 22)

前5行：


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,TenureGroup,IsMobileLogin
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,0-6个月,1
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,7-12个月,1
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,7-12个月,1
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,新用户,1
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,新用户,1



字段类型：


CustomerID                       int64
Churn                            int64
Tenure                         float64
PreferredLoginDevice               str
CityTier                         int64
WarehouseToHome                float64
PreferredPaymentMode               str
Gender                             str
HourSpendOnApp                 float64
NumberOfDeviceRegistered         int64
PreferedOrderCat                   str
SatisfactionScore                int64
MaritalStatus                      str
NumberOfAddress                  int64
Complain                         int64
OrderAmountHikeFromlastYear    float64
CouponUsed                     float64
OrderCount                     float64
DaySinceLastOrder              float64
CashbackAmount                 float64
TenureGroup                        str
IsMobileLogin                    int64
dtype: object


验收结果：


,0
行数,5630
列数,22
CustomerID重复数,0
核心字段缺失数,0
Churn取值,"[0, 1]"



数据粒度：一行代表一名用户，每个CustomerID唯一对应一位用户。


In [95]:
# 完成上一个单元后再运行本检查点
assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert df.shape == (5630, 22), "数据形状应为(5630, 22)"
assert df["CustomerID"].is_unique, "CustomerID应唯一"
assert set(df["Churn"].unique()) == {0, 1}, "Churn应只包含0和1"
print("检查点1通过")

# 数据粒度
print("\n数据粒度：一行代表一名用户，每个CustomerID唯一对应一位用户的基本信息和行为数据。")

检查点1通过

数据粒度：一行代表一名用户，每个CustomerID唯一对应一位用户的基本信息和行为数据。


In [96]:
# 任务2：公共基础指标（必做）

# TODO：构建overall_metrics DataFrame
overall_metrics = pd.DataFrame({
    "指标": [
        "用户数", "流失人数", "流失率", "平均订单数", "订单数中位数",
        "平均优惠券数", "平均返现", "平均App时长", "平均满意度", "平均距上次下单天数"
    ],
    "数值": [
        df["CustomerID"].nunique(),
        df["Churn"].sum(),
        df["Churn"].mean(),
        df["OrderCount"].mean(),
        df["OrderCount"].median(),
        df["CouponUsed"].mean(),
        df["CashbackAmount"].mean(),
        df["HourSpendOnApp"].mean(),
        df["SatisfactionScore"].mean(),
        df["DaySinceLastOrder"].mean(),
    ]
})

display(overall_metrics)

,指标,数值
0,用户数,"5,630.00"
1,流失人数,948.00
2,流失率,0.17
3,平均订单数,2.96
4,订单数中位数,2.00
5,平均优惠券数,1.72
6,平均返现,177.22
7,平均App时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.46


In [97]:
# 检查点2
assert isinstance(overall_metrics, pd.DataFrame), "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, "公共指标至少10项"

# TODO：将下面变量赋值为你计算的总体流失率
overall_churn_rate = df["Churn"].mean()
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, "总体流失率不正确"
print("检查点2通过")
print(f"总体流失率：{overall_churn_rate:.2%}")

检查点2通过
总体流失率：16.84%


In [98]:
# 任务3：单维专题分析（必做）

# 专题A：TenureGroup用户生命周期分析
segment_field = "TenureGroup"

# TODO：使用groupby + agg完成命名聚合
segment_analysis = (
    df.groupby(segment_field, observed=True)
      .agg(
          用户数=("CustomerID", "nunique"),
          流失人数=("Churn", "sum"),
          流失率=("Churn", "mean"),
          平均订单数=("OrderCount", "mean"),
          平均优惠券数=("CouponUsed", "mean"),
          平均返现=("CashbackAmount", "mean"),
          平均距上次下单天数=("DaySinceLastOrder", "mean"),
      )
      .reset_index()
      .sort_values("流失率", ascending=False)
)

# 添加用户占比
segment_analysis["用户占比"] = segment_analysis["用户数"] / len(df)

display(segment_analysis)

,TenureGroup,用户数,流失人数,流失率,平均订单数,平均优惠券数,平均返现,平均距上次下单天数,用户占比
4,新用户,508,272,0.54,1.89,0.96,142.44,2.88,0.09
0,0-6个月,1642,425,0.26,2.68,1.74,164.87,4.05,0.29
3,7-12个月,1584,156,0.10,2.75,1.60,163.31,4.38,0.28
1,13-24个月,1467,95,0.06,3.70,2.02,204.92,5.32,0.26
2,24个月以上,429,0,0.00,3.55,1.94,222.34,5.26,0.08


In [99]:
# 检查点3
assert segment_field in df.columns, "segment_field不是有效字段"
assert isinstance(segment_analysis, pd.DataFrame), "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, "专题表必须包含用户数"
assert len(segment_analysis) >= 2, "专题分析至少应有两个分组"
print("检查点3通过")

检查点3通过


In [100]:
# 任务4：双维度交叉分析（必做）

# TODO：选择两个维度
dim_1 = "TenureGroup"
dim_2 = "Complain"

# TODO：按两个维度统计
cross_analysis = (
    df.groupby([dim_1, dim_2], observed=True)
      .agg(
          用户数=("CustomerID", "nunique"),
          流失人数=("Churn", "sum"),
          流失率=("Churn", "mean"),
          平均订单数=("OrderCount", "mean"),
          平均满意度=("SatisfactionScore", "mean"),
      )
      .reset_index()
)

# TODO：新增"样本提示"列
cross_analysis["样本提示"] = np.where(cross_analysis["用户数"] < 30, "小样本", "可观察")

# 添加投诉状态标签
cross_analysis["投诉状态"] = cross_analysis[dim_2].map({0: "无投诉", 1: "有投诉"})

# TODO：按流失率排序
cross_analysis = cross_analysis.sort_values("流失率", ascending=False)

display(cross_analysis)

,TenureGroup,Complain,用户数,流失人数,流失率,平均订单数,平均满意度,样本提示,投诉状态
9,新用户,1,194,139,0.72,2.12,2.94,可观察,有投诉
1,0-6个月,1,465,236,0.51,2.87,3.03,可观察,有投诉
8,新用户,0,314,133,0.42,1.75,3.32,可观察,无投诉
7,7-12个月,1,406,81,0.20,2.67,3.02,可观察,有投诉
0,0-6个月,0,1177,189,0.16,2.61,3.11,可观察,无投诉
3,13-24个月,1,414,52,0.13,3.35,2.91,可观察,有投诉
6,7-12个月,0,1178,75,0.06,2.78,2.98,可观察,无投诉
2,13-24个月,0,1053,43,0.04,3.85,3.16,可观察,无投诉
5,24个月以上,1,125,0,0.00,3.06,3.19,可观察,有投诉
4,24个月以上,0,304,0,0.00,3.75,3.00,可观察,无投诉


In [101]:
# 检查点4
assert dim_1 in df.columns and dim_2 in df.columns, "两个维度必须是有效字段"
assert dim_1 != dim_2, "两个维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), "cross_analysis应为DataFrame"
assert {"用户数", "流失率", "样本提示"}.issubset(cross_analysis.columns), "双维表缺少必需列"
assert set(cross_analysis["样本提示"]).issubset({"小样本", "可观察"}), "样本提示取值不正确"
print("检查点4通过")

检查点4通过


In [102]:
# 任务5：报表输出与回读验证（必做）

# TODO：将三个表导出到OUTPUT_DIR
outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

# TODO：循环导出并重新读取
print("导出文件验证：")
for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    reloaded = pd.read_csv(path)
    print(f"  {filename}: 原始形状 {table.shape}, 回读形状 {reloaded.shape}, 一致: {reloaded.shape == table.shape}")

导出文件验证：
  overall_metrics.csv: 原始形状 (10, 2), 回读形状 (10, 2), 一致: True
  segment_analysis.csv: 原始形状 (5, 9), 回读形状 (5, 9), 一致: True
  cross_analysis.csv: 原始形状 (10, 9), 回读形状 (10, 9), 一致: True


In [103]:
# 检查点5
for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    assert path.exists(), f"缺少输出文件：{filename}"
    reloaded = pd.read_csv(path)
    assert reloaded.shape == table.shape, f"{filename}回读形状不一致"
print("检查点5通过：三个CSV均已成功导出并回读。")

检查点5通过：三个CSV均已成功导出并回读。


In [104]:
# 任务6：结论、限制与建议（必做）

print("=" * 60)
print("分析结论、限制与建议")
print("=" * 60)

print("\n### 结论1")
print("在新用户群体中，流失率为54%，与0-6个月用户(26%)和7-12个月用户(10%)相比显著更高。")
print("对应证据表：segment_analysis.csv (TenureGroup分组分析)")
print("说明：新用户（tenure=0）的流失率远高于其他任何用户群体，这表明用户早期体验对留存至关重要。")

print("\n### 结论2")
print("在有投诉的用户中，流失率为32%，而无投诉用户的流失率仅为11%。")
print("对应证据表：cross_analysis.csv (TenureGroup × Complain交叉分析)")
print("说明：投诉与流失存在强相关关系，有投诉用户的流失概率是无投诉用户的近3倍。")

print("\n### 结论3")
print("24个月以上用户流失率为0%，用户数为429人，占总体8%。")
print("对应证据表：segment_analysis.csv (TenureGroup分组分析)")
print("说明：长期用户的留存率极高，他们是平台最稳定的用户基础。")

print("\n### 分析限制")
print("1. 缺少订单金额和订单日期，因此无法计算GMV、客单价、复购周期或时间趋势分析。")
print("2. 数据为横截面数据，无法追踪用户行为变化，相关关系不等于因果关系。")
print("3. 投诉原因未记录，无法深入分析投诉的具体类型和严重程度。")

print("\n### 运营建议与验证方式")
print("建议：针对新用户群体，设计30天新手引导和首单优惠计划，降低早期流失率。")
print("验证方式：")
print("  - 需要获取用户注册日期和首次下单时间，验证新用户流失的关键时间窗口")
print("  - 建议设计A/B测试：对照组正常流程，实验组接受新手引导和首单优惠")
print("  - 对比两组的7天、14天、30天留存率差异")
print("  - 同时监控新用户投诉率变化，确保服务体验提升")

分析结论、限制与建议

### 结论1
在新用户群体中，流失率为54%，与0-6个月用户(26%)和7-12个月用户(10%)相比显著更高。
对应证据表：segment_analysis.csv (TenureGroup分组分析)
说明：新用户（tenure=0）的流失率远高于其他任何用户群体，这表明用户早期体验对留存至关重要。

### 结论2
在有投诉的用户中，流失率为32%，而无投诉用户的流失率仅为11%。
对应证据表：cross_analysis.csv (TenureGroup × Complain交叉分析)
说明：投诉与流失存在强相关关系，有投诉用户的流失概率是无投诉用户的近3倍。

### 结论3
24个月以上用户流失率为0%，用户数为429人，占总体8%。
对应证据表：segment_analysis.csv (TenureGroup分组分析)
说明：长期用户的留存率极高，他们是平台最稳定的用户基础。

### 分析限制
1. 缺少订单金额和订单日期，因此无法计算GMV、客单价、复购周期或时间趋势分析。
2. 数据为横截面数据，无法追踪用户行为变化，相关关系不等于因果关系。
3. 投诉原因未记录，无法深入分析投诉的具体类型和严重程度。

### 运营建议与验证方式
建议：针对新用户群体，设计30天新手引导和首单优惠计划，降低早期流失率。
验证方式：
  - 需要获取用户注册日期和首次下单时间，验证新用户流失的关键时间窗口
  - 建议设计A/B测试：对照组正常流程，实验组接受新手引导和首单优惠
  - 对比两组的7天、14天、30天留存率差异
  - 同时监控新用户投诉率变化，确保服务体验提升


In [105]:
# 完成确认
print("所有必做任务已完成！")
print("")
print(f"输出目录：{OUTPUT_DIR}")
print("")
print("已生成的报表：")
print("  1. overall_metrics.csv - 总体指标")
print("  2. segment_analysis.csv - 单维专题分析")
print("  3. cross_analysis.csv - 双维交叉分析")

所有必做任务已完成！

输出目录：C:\Users\21975\PyCharmMiscProject\output\day05_analysis

已生成的报表：
  1. overall_metrics.csv - 总体指标
  2. segment_analysis.csv - 单维专题分析
  3. cross_analysis.csv - 双维交叉分析


In [106]:
# 查看输出目录中的所有文件
print("输出目录内容：")
for file in OUTPUT_DIR.iterdir():
    if file.is_file():
        print(f"  - {file.name}")

输出目录内容：
  - category_analysis.csv
  - complain_analysis.csv
  - cross_analysis.csv
  - overall_metrics.csv
  - payment_analysis.csv
  - segment_analysis.csv
  - tenure_analysis.csv
  - tenure_complain_analysis.csv
  - tenure_complain_pivot.csv


In [107]:
# 清理旧的CSV文件（可选）
import os

# 删除旧的day5_analysis相关文件
for file in OUTPUT_DIR.glob("day5_analysis*.csv"):
    os.remove(file)
    print(f"已删除: {file.name}")

# 删除旧的day5_notebook相关文件
for file in OUTPUT_DIR.glob("day5_notebook*.csv"):
    os.remove(file)
    print(f"已删除: {file.name}")

print("")
print("清理完成！")


清理完成！


In [108]:
# 重新生成规范的三个CSV文件
print("重新生成规范的三个CSV文件：")
print("")

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"已生成: {filename}")

print("")
print("生成的规范文件：")
for file in OUTPUT_DIR.glob("*.csv"):
    print(f"  - {file.name}")

重新生成规范的三个CSV文件：

已生成: overall_metrics.csv
已生成: segment_analysis.csv
已生成: cross_analysis.csv

生成的规范文件：
  - category_analysis.csv
  - complain_analysis.csv
  - cross_analysis.csv
  - overall_metrics.csv
  - payment_analysis.csv
  - segment_analysis.csv
  - tenure_analysis.csv
  - tenure_complain_analysis.csv
  - tenure_complain_pivot.csv
